# Generate AI reviews with OPEN models - domain-broadening (Route B)
Machine reviews for the non-product domains (**movies, restaurants, apps, books, games**)
with **Qwen2.5-7B + Llama-3-8B + Zephyr-7B** (Qwen / Llama / Mistral families). Pairs against the human reviews we already have per domain.

### Instructions
1. **Runtime -> A100 GPU** (T4: set `LOAD_4BIT=True`).
2. Run all cells.
3. Download `ai_generated_reviews_domains_open.csv` -> put in project `data/`.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import os, csv, random, gc
import torch
from transformers import pipeline, BitsAndBytesConfig

SEED = 42; random.seed(SEED)
MODELS = [("Qwen/Qwen2.5-7B-Instruct","qwen"),
          ("NousResearch/Meta-Llama-3-8B-Instruct","llama3"),
          ("HuggingFaceH4/zephyr-7b-beta","zephyr")]   # Mistral family (ungated, native); Phi-3 dropped
PER_DOMAIN_PER_MODEL = 500     # 5 domains x 3 models x 500 = ~7,500
BATCH = 16
LOAD_4BIT = False              # True on T4

PERSONAS = ["a busy parent","a college student","a retiree","a casual fan","a hardcore enthusiast",
            "a first-timer","a longtime regular","a skeptic","a bargain hunter","a critic at heart"]
LENGTHS = ["in just one short sentence","in one or two sentences","in a short paragraph",
           "in a detailed paragraph or two"]
STYLES = ["polished and well-written","casual and conversational",
          "quick and informal, all lowercase with a couple of typos like a phone review",
          "enthusiastic and full of energy","measured, balanced and a little critical","blunt and to the point"]
STARS = [5,5,5,5,4,4,4,3,2,1,1]
DOMAINS = {
  "movies":      ("you watched", ["a sci-fi movie","a romantic comedy","a horror film","an action blockbuster",
                  "a slow indie drama","an animated family movie","a documentary","a psychological thriller",
                  "a superhero movie","a foreign-language film"]),
  "restaurants": ("you visited", ["an Italian restaurant","a sushi spot","a taco truck","a fine-dining steakhouse",
                  "a vegan cafe","a burger joint","a Thai restaurant","a brunch place","a pizzeria","a diner","a ramen shop"]),
  "apps":        ("you've been using", ["a budgeting app","a fitness-tracking app","a photo-editing app",
                  "a language-learning app","a meditation app","a food-delivery app","a note-taking app",
                  "a ride-share app","a dating app","a streaming app"]),
  "books":       ("you read", ["a mystery novel","a fantasy epic","a self-help book","a historical romance",
                  "a sci-fi novel","a memoir","a literary thriller","a cookbook","a young-adult novel","a history book"]),
  "games":       ("you played", ["an open-world RPG","an indie platformer","a first-person shooter","a cozy farming sim",
                  "a roguelike","a racing game","a puzzle game","a survival-horror game","a strategy game","a fighting game"]),
}
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
def build_prompt(domain):
    verb, subjects = DOMAINS[domain]
    subject = random.choice(subjects)
    persona = random.choice(PERSONAS); length = random.choice(LENGTHS)
    style = random.choice(STYLES); stars = random.choice(STARS)
    user = (f"Write a realistic online review of {subject} {verb}, as if you are {persona}, "
            f"giving {stars} out of 5 stars. Write it {style}. Invent a plausible specific "
            f"title/name and concrete details, sound like a genuine human - do NOT include the "
            f"star rating, a header, or any preamble. Just output the review text.")
    return user, persona, style, stars, subject

In [ ]:
OUT = "ai_generated_reviews_domains_open.csv"
new = not os.path.exists(OUT)
f = open(OUT, "a", newline="", encoding="utf-8"); w = csv.writer(f)
if new: w.writerow(["text","domain","source","model_id","rating","persona","style","subject"])
written = 0

for model_id, fam in MODELS:
    print("\n=== loading", model_id, "===")
    kw = dict(torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    if LOAD_4BIT:
        kw = dict(model_kwargs={"quantization_config": BitsAndBytesConfig(
                      load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},
                  device_map="auto", trust_remote_code=True)
    try:
        pipe = pipeline("text-generation", model=model_id, **kw)
    except Exception as e:
        print("  load failed, skipping:", e); continue
    tok = pipe.tokenizer
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "left"

    for domain in DOMAINS:
        prompts, meta = [], []
        for _ in range(PER_DOMAIN_PER_MODEL):
            user, persona, style, stars, subject = build_prompt(domain)
            prompts.append([{"role":"user","content":user}]); meta.append((domain,stars,persona,style,subject))
        for i in range(0, len(prompts), BATCH):
            batch = prompts[i:i+BATCH]
            try:
                outs = pipe(batch, max_new_tokens=256, do_sample=True, temperature=0.85, top_p=0.95,
                            batch_size=len(batch), pad_token_id=tok.pad_token_id)
            except Exception as e:
                print("  gen error:", e); continue
            for j, o in enumerate(outs):
                rec = o[0] if isinstance(o, list) else o
                gen = rec["generated_text"]
                text = gen[-1]["content"] if isinstance(gen, list) else str(gen)
                text = text.strip().replace("\n", " ")
                dm, stars, persona, style, subject = meta[i+j]
                if len(text.split()) >= 4:
                    w.writerow([text, dm, fam, model_id, stars, persona, style, subject]); written += 1
            f.flush()
        print(f"  {fam} / {domain}: done (total {written})")
    del pipe; gc.collect(); torch.cuda.empty_cache()
    # free DISK: remove this model's downloaded weights from the HF cache so the
    # next model's download doesn't fill the Colab disk (~15GB each).
    import shutil
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" +
                  model_id.replace("/", "--")), ignore_errors=True)
    print(fam, "done (weights cache freed)")
f.close()
print("\nTOTAL written:", written)

In [ ]:
from google.colab import files
files.download("ai_generated_reviews_domains_open.csv")

## Next steps
Put `ai_generated_reviews_domains_open.csv` in `data/` and tell the assistant. It will
add these as CG sources (per domain + family), add the matching human OR sources
(IMDB/Yelp/apps/Amazon-books/Steam), rebuild, re-run LOGO + leave-one-domain-out, then
retrain everything once.